In [0]:
class VectorSearch:
    def __init__(self):
        from databricks.vector_search.client import VectorSearchClient
        self.vs_client = VectorSearchClient(disable_notice = True)

    def create_endpoint(self, endpoint_name):
        eps = self.vs_client.list_endpoints()
        if len(eps)==0 or endpoint_name not in [ep["name"] for ep in eps["endpoints"]]:
            print(f"Creating Vector Endpoint...", end="")
            self.vs_client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
            print("Done")

    def delete_endpoint(self, endpoint_name):
        eps = self.vs_client.list_endpoints()
        if len(eps)>0 and endpoint_name in [ep["name"] for ep in eps["endpoints"]]:
            print(f"Deleting Vector Endpoint...", end="")
            self.vs_client.delete_endpoint(name=endpoint_name)
            print("Done")

    def create_index(
        self,
        index_name,
        source_table_name,
        primary_key,
        embedding_source_column,
        embedding_model_endpoint_name,
        vector_endpoint_name,
    ):
        idxs = self.vs_client.list_indexes(vector_endpoint_name)
        if len(idxs)==0 or index_name not in [idx["name"] for idx in idxs["vector_indexes"]]:
            print(f"Creating Index...", end="")
            self.vs_client.create_delta_sync_index(
                index_name=f"{index_name}",
                source_table_name=f"{source_table_name}",
                primary_key=primary_key,
                embedding_source_column=embedding_source_column,
                embedding_model_endpoint_name=embedding_model_endpoint_name,
                pipeline_type="TRIGGERED",
                endpoint_name=vector_endpoint_name,
            )
            print("Done")

    def sync_index(self, index_name):
        import time
        print(f"Syncing Index...", end="")
        index = self.vs_client.get_index(index_name=f"{index_name}")
        index.sync()
        idx_status = False
        while idx_status!=True:
            time.sleep(10)
            idx_status = index.describe()["status"]["ready"]
        print("Done")

    def delete_index(self, index_name, vector_endpoint_name):
        eps = self.vs_client.list_endpoints()
        if len(eps)> 0 and vector_endpoint_name in [ep["name"] for ep in eps["endpoints"]]:
            idxs = self.vs_client.list_indexes(vector_endpoint_name)
            if len(idxs)>0 and index_name in [idx["name"] for idx in idxs["vector_indexes"]]:
                print(f"Deleting Index...", end="")
                self.vs_client.delete_index(index_name=f"{index_name}")
                print("Done")
        
